# Cost-Aware Simulation Based Inference

This notebook demonstrates the use of the `CostAwareSimulator` in BayesFlow. 

Generating samples for simulation-based inference can have a high computational cost, and the cost of an individual simulation can vary depending on the parameters used for simulation. Cost-aware simulation based inference (SBI) uses importance sampling to encourage sampling from the computationally cheaper parameterisations of the model. In this way, it can significantly reduce simulation costs, without any changes to the simulator itself, while still allowing for unbiased inference through the use of importance weights.

In this notebook, we will:
1. Define a cost model that mimics the computational cost of a simulator.
2. Use `CostAwareSimulator` to sample cost-efficient parameters from a prior.
3. Evaluate the performance gain using Effective Sample Size (ESS) and Computational Gain (CG).
4. Perform inference using the cost-aware samples.
5. Explore Multiple Importance Sampling (MIS) to handle cases where the posterior may lie in high-cost regions.

**References:**
- Bharti et al. (2025) — [Cost-aware simulation-based inference](https://arxiv.org/abs/2410.07930)



## 1. Imports

In [ ]:
import os
os.environ["KERAS_BACKEND"] = "torch"

import numpy as np

import matplotlib.pyplot as plt
import bayesflow as bf



## 2. Input Cost Data as a function of the simulator parameters

In a real scenario, cost can be any value that is meaningful for the simulator, for example the wall-clock time of the simulation, which may depend on the parameters used. A function can be constructed by interpolating existing simulated data, for example, by fitting a Gaussian process to the data. These samples can then be reused later.  

In this toy example, we use an analytic function of the parameters for the SIR model ($\beta ,\gamma$) to mimic the cost behaviour of the model. 

In [ ]:
def cost_model(theta: np.ndarray) -> np.ndarray:
    """
    An analytic cost model that predicts the cost as a function of input parameters theta.
    
    Parameters
    ---------- 
    theta : np.ndarray
        The parameter values to predict the cost for. 
        Shape (m,) or (m, d).

    Returns
    -------
    cost : np.ndarray
        The cost, with shape (m,).
    """
                                                                                                                            
    theta = np.asarray(theta, dtype=float)                                                                                                   
                                                                                                                                            
    if theta.ndim == 1:                                                                                                                      
        # Cost is high (near 1.0) when theta is small, decays to 0 as theta grows                                                            
        cost = np.exp(-np.abs(theta))                                                                                                        
    else:                                                                                                                                    
        # Sum of decays across dimensions                                                                                                    
        cost = np.sum(np.exp(-np.abs(theta)), axis=1)                                                                                        
                                                                                                                                            
    return cost             


We can visualise how the cost changes with the parameters $\beta$ and $\gamma$. 

In [ ]:

def plot_cost_landscape(cost_model):
    #add predicted cost plot

    # Create a grid for cost visualization
    b_min, b_max = 0, 3.5 # Approximate ranges for SIR beta
    g_min, g_max = 0, 0.35   # Approximate ranges for SIR gamma
    B, G = np.meshgrid(np.linspace(b_min, b_max, 50), np.linspace(g_min, g_max, 50))
    grid_theta = np.stack([B.ravel(), G.ravel()], axis=1)

    predicted_costs = cost_model(grid_theta)
    COST = predicted_costs.reshape(B.shape)

    fig = plt.figure(figsize=(6, 4))
    plt.xlim((b_min, b_max))
    plt.ylim((g_min, g_max))
    plt.pcolormesh(B, G, COST, cmap="viridis", shading="auto")
    plt.colorbar(label="Predicted Cost")
    plt.xlabel(r"$\beta$ (Contact Rate)")
    plt.ylabel(r"$\gamma$ (Recovery Rate)")
    plt.title("Predicted Cost Heatmap")
    #plt.show()
    return fig 

fig = plot_cost_landscape(cost_model)
plt.show()

## 3. Initialize Cost-Aware Simulator

To implement cost-aware sampling, we use the `CostAwareSimulator`, which acts as a wrapper around a prior sampler. It uses the previously defined `cost_model` as a plugin to filter samples from the prior based on a regularized cost function $g(c(\theta))$. The cost is regularized by                                

\begin{equation}                                                                                                                             
    g(c) = \max\left(g_{\min}, (c + g_{\min})^k\right)                                                                                       
\end{equation}    

where $g_{\min}$ is a minimum value (offset) that prevents the regularized cost from becoming too small, which stops the acceptance probabilities in rejection sampling from becoming too small. The parameter $k$ acts as a penalty exponent (or regularization power), controlling how aggressively high-cost samples are suppressed. $g_{\min}$ is configurable through the class properties of the `CostAwareSimulator`, while $k$ can be specified when calling the `sample` method.

To avoid executing the computationally expensive simulator before we have selected cost-effective prior samples, the `CostAwareSimulator` accepts a prior function as input and initially returns only filtered samples from the prior. 

In [ ]:

#We define a simulator for the example. This will be used to generate samples once the prior parameters are found
sir_sim = bf.simulators.benchmark_simulators.SIR()

#define the prior to use for cost-aware sampling
prior = sir_sim.prior 

#initialise the cost-aware simulator 
cost_aware_sim = bf.simulators.CostAwareSimulator(prior=prior, cost_model=cost_model)

## 4. Cost-Aware Sampling

We use rejection sampling to obtain a set of cost-efficient parameters.

Candidates $\theta$ drawn from the prior distribution are accepted based on the regularized cost $g(c)$. To bias the sampling towards low-cost regions while maintaining a known relationship to the original prior, we define the acceptance probability as 
\begin{equation}                                                                                                                             
       P(\text{accept} \mid \theta) = \frac{g_{\min}}{g(c(\theta))}                                                                             
   \end{equation}   

This specific form ensures that samples with the lower cost are more likely to be accepted, while those with higher costs are accepted with decreasing probability. This effectively transforms the prior into a distribution that favors computationally cheaper regions.

Since the prior function is cheap to sample, the `sample` method only returns the requested number of samples.

In [ ]:
print("\nTesting cost-aware sampling...")
num_samples = 1000

#Use the cost-aware sampling method
accepted_samples = cost_aware_sim.sample(batch_shape=(num_samples,1))
accepted_theta = accepted_samples.get("parameters")

#number of samples
print(f"Successfully sampled {len(accepted_theta)} cost-efficient parameters.")


The samples generated by cost-aware sampling can be compared with those taken directly from the prior. 

In [ ]:

# Generate some samples directly for the prior, for comparison
prior_samples = np.array([prior() for _ in range(len(accepted_theta))])

def plot_prior_samples(cost_model,prior_samples,accepted_theta):
   
    fig = plot_cost_landscape(cost_model)
        
    plt.scatter(
        prior_samples[:, 0],
        prior_samples[:, 1],
        color="white",
        alpha=0.3,
        s=50,
        label="Prior Samples",
        marker=".",
    )

    # Accepted samples
    if len(accepted_theta) > 0:
        plt.scatter(
            accepted_theta[:, 0],
            accepted_theta[:, 1],
            color="red",
            edgecolors="white",
            s=50,
            label="Cost-Aware Samples",
            marker="o",
        )

    plt.xlabel(r"$\beta$ (Contact Rate)")
    plt.ylabel(r"$\gamma$ (Recovery Rate)")
    plt.title("Cost Landscape and Sample Distribution")
    plt.legend()
    return fig

fig = plot_prior_samples(cost_model,prior_samples,accepted_theta)
plt.show()

We can see how the distribution of the parameters has changed by favouring cost-effective options.

In [ ]:

def plot_histograms(prior_samples, accepted_theta, legend_labels=None):
    if legend_labels is None:
        legend_labels = ["Prior", "Cost-Aware"]

    # Visualise the change in parameter distributions
    plt.figure(figsize=(6, 3))

    # Distribution of first parameter
    plt.subplot(1, 2, 1)
    plt.hist(prior_samples[:, 0], bins=20, alpha=0.5, label=legend_labels[0], color="gray", density=True)
    if len(accepted_theta) > 0:
        plt.hist(accepted_theta[:, 0], bins=20, alpha=0.7, label=legend_labels[1], color="red", density=True)
    plt.xlabel(r"$\beta$")
    plt.ylabel("Density")
    plt.title(r"Distribution of $\beta$")
    plt.legend()

    # Distribution of second parameter
    plt.subplot(1, 2, 2)
    plt.hist(prior_samples[:, 1], bins=20, alpha=0.5, label=legend_labels[0], color="gray", density=True)
    if len(accepted_theta) > 0:
        plt.hist(accepted_theta[:, 1], bins=20, alpha=0.7, label=legend_labels[1], color="red", density=True)
    plt.xlabel(r"$\gamma$")
    plt.ylabel("Density")
    plt.title(r"Distribution of $\gamma$")
    plt.legend()

    plt.tight_layout()

    return fig

fig = plot_histograms(prior_samples, accepted_theta)

plt.show()

## 5. Performance Metrics

We evaluate the effectiveness of the cost-aware sampling using Effective Sample Size (ESS) and Computational Gain (CG). Together, these metrics evaluate the trade-off between the statistical quality of the samples and the computational effort saved.

                                                                   
   \begin{equation}                                                                                                                             
       \text{ESS} = \frac{\left( \sum_{i=1}^{N} g(c(\theta_i)) \right)^2}{N \sum_{i=1}^{N} g(c(\theta_i))^2}                                                          
   \end{equation}         


In [ ]:
metrics = cost_aware_sim.compute_metrics({"theta": accepted_theta})
print(f"  Effective Sample Size: {metrics['ess']:.2f}")

Computational gain measures the difference in the expected cost (defined by the cost function) by using the cost-aware samples over plainly sampling the prior. This metric is the ratio of the average cost of samples drawn from the original prior over the average cost of the samples accepted by the cost-aware sampler.


\begin{equation*}                 
\text{CG} = \frac{\mathbb{E}_{\theta \sim \pi(\theta)} (c(\theta))}{\mathbb{E}_{\theta \sim \pi_{\text{aware}}(\theta)} (c(\theta))}    
\end{equation*}                                        
       
This is estimated as                 
\begin{equation*}                                              \text{CG} \approx \frac{\frac{1}{N} \sum_{j=1}^{N} c(\theta_j^{\text{prior}})}{\frac{1}{N} \sum_{i=1}^{N} c(\theta_i^{\text{accepted}})}
\end{equation*}     
where samples $\theta_j^{\text{prior}}$ are taken from the original prior distribution. 


In [ ]:
print(f"  Computational Gain:  {metrics['cg']:.2f}")

## 6. Generating Expensive Samples

Now that we have filtered out for cost-efficient values of the prior parameters, we can finally sample the expensive simulator.

In [ ]:

accepted_theta = accepted_samples["parameters"]

train_observations_cost_aware = np.array([sir_sim.observation_model(t) for t in accepted_theta])
print(f"\nSuccessfully simulated {len(train_observations_cost_aware)} samples using the expensive simulator.")


## 7. Computing Importance Weights

We use self-normalised weights $w_i$ for the i-th accepted sample $\theta_i$. Let $N$ be the total number of accepted samples in the batch. 
\begin{equation}                                                                                                                             
       w_i = \frac{g(c(\theta_i))}{\sum_{j=1}^{N} g(c(\theta_j))}                                                                               
   \end{equation}                                                                

In [ ]:

train_weights = cost_aware_sim.compute_weights(accepted_samples)

train_weights = np.asarray(train_weights, dtype=np.float32).reshape(-1, 1)


## 8. Inference

We can now plug the training data into the existing Bayesflow framework. The training weights are passed to the workflow via an adapter

In [ ]:

train_data_cost_aware = {
    "theta": np.asarray(accepted_theta),
    "observations": np.asarray(train_observations_cost_aware),
    "weights": train_weights}

adapter = (
    bf.Adapter()
    .to_array()
    .convert_dtype("float64", "float32")
    .concatenate(["theta"], into="inference_variables")
    .concatenate(["observations"], into="inference_conditions")
    .rename("weights", "sample_weight")
)

inference_net = bf.networks.PointNetwork(points="mean")

workflow = bf.BasicWorkflow(
    simulator=cost_aware_sim,
    adapter=adapter,
    inference_network=inference_net,
)

We train the model with cost-aware data

In [ ]:
history_cost_aware = workflow.fit_offline(                                                                                                                               
       data=train_data_cost_aware,                                                                                                                                
       epochs=20,                                                                                                                                     
       batch_size=32                                                                                                                                   
   )


## 9. Multiple Importance sampling 
When the true posterior lies in the computationally costly region, using a single penalty exponent $k$ can be problematic: a small $k$ may not save enough cost, while a large $k$ might completely suppress the regions of the parameter space that are most informative for the inference. Single-$k$ sampling can therefore make it difficult to fully recover the prior.

To mitigate this, we can use Multiple Importance Sampling (MIS) by using several cost-aware priors with different values for the penalty exponent $k$. This gives a better coverage of the parameter space, because it balances cost efficiency without restricting the high-cost regions if they are posterior-dense.

The `CostAwareSimulator` supports multiple importance sampling by accepting an array of $k$ values in the sample method. The simulator executes the rejection sampling process for each value of $k$ provided; the total number of samples requested via `sample` is split equally among these values. The resulting weights are then self-normalized relative to each $k$-group.  
 

In [ ]:
regularization_powers = [0,2,3,5]

accepted_samples_mis = cost_aware_sim.sample(batch_shape=(num_samples,1),k=regularization_powers)

fig = plot_prior_samples(cost_model,prior_samples,accepted_samples_mis["parameters"])
plt.show()

In [ ]:
accepted_theta_mis = accepted_samples_mis["parameters"]


fig = plot_histograms(prior_samples, accepted_theta_mis, legend_labels=["Cost-aware one k","MIS"])
fig.suptitle("Comparison of Samples from the Prior with Mutliple Importance Sampling and one k value")
plt.show()

In [ ]:

train_observations_mis = np.array([sir_sim.observation_model(t) for t in accepted_theta_mis])
print(f"\nSuccessfully simulated {len(train_observations_mis)} samples using the expensive simulator.")

#compute the weights using accepted samples
train_weights_mis = cost_aware_sim.compute_weights(accepted_samples_mis)
train_weights_mis = np.asarray(train_weights_mis, dtype=np.float32).reshape(-1, 1)

#setup the training data
train_data_mis = {
    "theta": np.asarray(accepted_theta_mis),
    "observations": np.asarray(train_observations_mis),
    "weights": train_weights_mis}

adapter_mis = (
    bf.Adapter()
    .to_array()
    .convert_dtype("float64", "float32")
    .concatenate(["theta"], into="inference_variables")
    .concatenate(["observations"], into="inference_conditions")
    .rename("weights", "sample_weight")
)

inference_net_mis = bf.networks.PointNetwork(points="mean")

workflow_mis = bf.BasicWorkflow(
    simulator=cost_aware_sim,
    adapter=adapter_mis,
    inference_network=inference_net_mis,
)

history_mis = workflow_mis.fit_offline(                                                                                                                               
       data=train_data_mis,                                                                                                                                
       epochs=20,                                                                                                                                     
       batch_size=32                                                                                                                                   
   )